## Imports

In [ ]:
from qsopt import * 
import numpy as np
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

## Define experimental parameters

In [2]:
# Define custom physical constants
custom_constants = PhysicalConstants(
    chi=0.75,                    # Dispersive coupling
    photon_cavity_coupling=1.5,  # Photon-cavity coupling
    inverse_pulse_width=0.2      # Inverse pulse width
)

# Define custom system dimensions
custom_dims = SystemDimensions(
    cavity_levels=2,
    qubit_levels=2,
    field_levels=2
)

# Define measurement protocol
custom_measurement = MeasurementProtocol(
    measurement_times = [-5.0, 0.0, 5.0]
)

# Define initial state configuration (SINGLE_PHOTON)
initial_state = InitialStateConfig(
    state_type=InitialStateType.SINGLE_PHOTON
)

# Define noise configuration
noise_config = NoiseConfiguration(
    depolarizing=0.01,  
    dephasing=0.005,      
    relaxation=0.01     
)

# Create parameters with custom configuration
exp_parameters = ExperimentalParameters(
    physical_constants=custom_constants,
    system_dims=custom_dims,
    measurement=custom_measurement,
    initial_state=initial_state,
    noise_config=noise_config
)

print(exp_parameters)


SYSTEM DIMENSIONS
------------------------------
  Cavity levels:             2
  Qubit levels:              2
  Field levels:              2
  Total dimension:           8
  Status:               VALID

PHYSICAL CONSTANTS
------------------------------
  Chi:                    0.7500
  Photon cavity coupling: 1.5000
  Inverse pulse width:    0.2000
  Status:               VALID

MEASUREMENT PROTOCOL
------------------------------
  Number of measurements:      3
  Measurement times: [-5.0, 0.0, 5.0]
  Status:               VALID

INITIAL STATE
------------------------------
  Type:                 single_photon

NOISE MODEL
------------------------------
  Depolarizing rate:      0.0100
  Dephasing rate:         0.0050
  Relaxation rate:        0.0100
  Total noise rate:       0.0250
  Custom operators:     None
  Status:               VALID

SYSTEM STATUS
------------------------------
  Configuration:        VALID


In [3]:
parameters = TrainableParameters()
parameters.add_rotation_angles(['ry1', 'ry2'], [1., 1.], optimizer=optax.adam(0.01))

print(parameters)

TrainableParameters(total=2, rotation_angles=2, measurement_times=0, custom=0)


In [4]:
experiment = SingleQubitExperiment(exp_parameters, parameters)


## Test Single Simulation

Before optimization, let's run a single simulation to see the initial state of the system.

In [ ]:
# Use the new run_simulation method - much simpler!
# This method automatically uses the current parameter values from trainable_params

results = experiment.run_simulation()

print(f"Initial state:")
print(f"  Shape: {experiment.get_initial_state().shape}")
print(f"  Trace: {experiment.get_initial_state().tr():.6f}")

print(f"\nCurrent parameters:")
print(f"  θ₁ = {results['theta1']:.4f} rad")
print(f"  θ₂ = {results['theta2']:.4f} rad")

print(f"\nSimulation results:")
print(f"  P(with interaction):    {results['prob_with']:.6f}")
print(f"  P(without interaction): {results['prob_without']:.6f}")
print(f"  Contrast:               {results['contrast']:.6f}")

# Store initial contrast for comparison later
initial_contrast = results['contrast']

## Run Optimization

Now let's optimize the parameters to maximize the detection contrast.

## Setup Callback

Create a callback to track optimization metrics including loss, contrast, and detection probabilities.

In [ ]:
# Create a callback to track optimization progress
callback = OptimizationCallback(save_every=1, save_best=True)

print(f"Callback initialized: {callback}")

In [ ]:
# Run optimization for 20 steps with callback
print("Starting optimization...\n")

history = experiment.optimize(
    num_steps=20,
    learning_rate=0.05,
    verbose=True,
    callback=callback  # Pass the callback to track metrics
)

print("\nOptimization complete!")
print(f"Callback recorded {len(callback.history['epochs'])} epochs")

## Visualize Optimization Progress

Use the callback data to plot the optimization trajectory.

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Contrast vs Epoch
ax = axes[0, 0]
ax.plot(callback.history['epochs'], callback.history['contrast'], 'b-', linewidth=2, label='Contrast')
ax.axhline(y=callback.best_metrics['contrast'], color='r', linestyle='--', label='Best contrast')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Contrast', fontsize=12)
ax.set_title('Sensing Contrast Evolution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Loss vs Epoch
ax = axes[0, 1]
ax.plot(callback.history['epochs'], callback.history['loss'], 'r-', linewidth=2, label='Loss')
ax.axhline(y=callback.best_loss, color='g', linestyle='--', label='Best loss')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Function Evolution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Detection Probabilities
ax = axes[1, 0]
ax.plot(callback.history['epochs'], callback.history['prob_with'], 'g-', linewidth=2, label='P(with photon)')
ax.plot(callback.history['epochs'], callback.history['prob_without'], 'orange', linewidth=2, label='P(without photon)')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Detection Probabilities', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Parameters trajectory
ax = axes[1, 1]
params_array = np.array(callback.history['parameters'])
ax.plot(callback.history['epochs'], params_array[:, 0], 'b-', linewidth=2, label='θ₁')
ax.plot(callback.history['epochs'], params_array[:, 1], 'r-', linewidth=2, label='θ₂')
best_params = callback.get_best_parameters()
ax.axhline(y=best_params[0], color='b', linestyle='--', alpha=0.5)
ax.axhline(y=best_params[1], color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Parameter Value (rad)', fontsize=12)
ax.set_title('Parameter Trajectories', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest metrics found at epoch {callback.best_metrics['epoch']}:")
print(f"  Best contrast: {callback.best_metrics['contrast']:.6f}")
print(f"  Best loss: {callback.best_loss:.6f}")
print(f"  Best parameters: θ₁={best_params[0]:.4f}, θ₂={best_params[1]:.4f}")

## Save and Load Callback Data

Save the optimization results for future analysis and plotting.

In [ ]:
# Save callback results
callback.save('optimization_results.npz')
print("Saved optimization results to 'optimization_results.npz'")

# Load the results (demonstrating how to reload for future use)
loaded_data = OptimizationCallback.load('optimization_results.npz')

print("\nLoaded data contains:")
for key in loaded_data.keys():
    if hasattr(loaded_data[key], 'shape'):
        print(f"  {key}: shape {loaded_data[key].shape}")
    else:
        print(f"  {key}: {loaded_data[key]}")

# Example: Plot contrast from loaded data
plt.figure(figsize=(8, 5))
plt.plot(loaded_data['epochs'], loaded_data['contrast'], 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Contrast')
plt.title('Sensing Contrast (from loaded data)')
plt.grid(True, alpha=0.3)
plt.show()

## View Results

In [ ]:
# Get final values directly from trainable_params (they're updated during optimization)
final_results = experiment.run_simulation()

print("="*60)
print("OPTIMIZATION SUMMARY")
print("="*60)
print(f"\nInitial Parameters:")
print(f"  θ₁ = {np.pi/2:.4f} rad = {np.degrees(np.pi/2):.2f}°")
print(f"  θ₂ = {-np.pi/2:.4f} rad = {np.degrees(-np.pi/2):.2f}°")
print(f"  Contrast = {initial_contrast:.6f}")

print(f"\nFinal Parameters:")
print(f"  θ₁ = {final_results['theta1']:.4f} rad = {np.degrees(final_results['theta1']):.2f}°")
print(f"  θ₂ = {final_results['theta2']:.4f} rad = {np.degrees(final_results['theta2']):.2f}°")
print(f"  Contrast = {final_results['contrast']:.6f}")

if abs(initial_contrast) > 1e-10:
    improvement = ((final_results['contrast'] - initial_contrast) / abs(initial_contrast)) * 100
    print(f"\nImprovement: {improvement:+.2f}%")
else:
    print(f"\nImprovement: N/A (initial contrast near zero)")
print("="*60)

## Manually Update Parameters

You can also manually update parameter values and re-run simulations:

In [ ]:
# Update parameters directly in the trainable_params object
parameters.parameters[0].value = 0.0  # Set θ₁ to 0
parameters.parameters[1].value = np.pi  # Set θ₂ to π

# Run simulation with new values
new_results = experiment.run_simulation()

print(f"Testing with θ₁ = {new_results['theta1']:.4f}, θ₂ = {new_results['theta2']:.4f}")
print(f"Contrast: {new_results['contrast']:.6f}")

# Restore optimized values
parameters.parameters[0].value = final_results['theta1']
parameters.parameters[1].value = final_results['theta2']
print(f"\nRestored optimized parameters")